# Волновое уравнение

## Базовый уровень

Решаем волновое уравнение

$$\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2},$$

с начальными условиями

$$u(0, x) = f(x), \quad \frac{\partial u(0, x)}{\partial t} = g(x).$$

На границе ставим граничные условия Дирихле (закреплённые концы):

$$u(t, 0) = 0, \quad u(t, L) = 0.$$

Дискретизация (схема "крест"):

$$\frac{u^{n+1}_i - 2 u^n_i + u^{n-1}_i}{\Delta t^2} = c^2 \frac{u^n_{i-1} - 2 u^n_i + u^n_{i+1}}{\Delta x^2},$$

$$u^{n+1}_i - 2 u^n_i + u^{n-1}_i = c^2 \frac{\Delta t^2}{{\Delta x^2}} \left(u^n_{i-1} - 2 u^n_i + u^n_{i+1}\right).$$

Введём переменную

$$r = c \frac{\Delta t}{{\Delta x}},$$

перенесём члены с предыдущего слоя в правую часть:

$$u^{n+1}_i = 2 u^n_i - u^{n-1}_i + r^2 \left(u^n_{i-1} - 2 u^n_i + u^n_{i+1}\right).$$

Граничные условия дискретизируются как:

$$u_1 = 0, \quad u_N = 0.$$

На первом слое используется схема

$$\frac{\frac{u^1_i - u^0_i}{\Delta t} - g(x_i)}{\Delta t} = c^2 \frac{u^0_{i-1} - 2 u^0_i + u^0_{i+1}}{\Delta x^2},$$

$$\frac{u^1_i - u^0_i}{\Delta t} - g(x_i) = c^2 \Delta t \left(\frac{u^0_{i-1} - 2 u^0_i + u^0_{i+1}}{\Delta x^2}\right),$$

$$u^1_i = u^0_i + g(x_i) \Delta t + c^2 \frac{\Delta t^2}{\Delta x^2} \left(u^0_{i-1} - 2 u^0_i + u^0_{i+1}\right),$$

$$u^1_i = u^0_i + g(x_i) \Delta t + r^2 \left(u^0_{i-1} - 2 u^0_i + u^0_{i+1}\right).$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

In [ ]:
def triangle(x, x0=0.5, half_width=0.1, amplitude=1.0):
    y = np.maximum(0, 1 - np.abs(x - x0) / half_width)
    return amplitude * y

In [ ]:
def simulate_wave(r, L=1.0, c=1.0, Nx=151, T=0.6, half_width=0.08, max_frames=120):
    dx = L / (Nx - 1)
    dt = r * dx / c
    Nt = int(np.ceil(T / dt))
    x = np.linspace(0, L, Nx)
    u0 = triangle(x, x0=L/2, half_width=half_width, amplitude=1.0)
    u_prev = u0.copy()
    u_curr = u_prev.copy()
    r2 = (c * dt / dx) ** 2
    
    # первый шаг
    for i in range(1, Nx-1):
        u_curr[i] = u_prev[i] + 0.5 * r2 * (u_prev[i+1] - 2 * u_prev[i] + u_prev[i-1])
    u_curr[0] = u_curr[-1] = 0.0
    
    save_every = max(1, int(np.ceil(Nt / max_frames)))
    
    history = [u_prev.copy()]

    for n in range(1, Nt):
        u_next = np.zeros_like(u_curr)
        for i in range(1, Nx-1):
            u_next[i] = 2*u_curr[i] - u_prev[i] + r2 * (u_curr[i+1] - 2*u_curr[i] + u_curr[i-1])
        u_next[0] = u_next[-1] = 0.0
        if n % save_every == 0:
            history.append(u_next.copy())
        u_prev, u_curr = u_curr, u_next
    return x, history

In [ ]:
def make_animation(x, history, title=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    line, = ax.plot(x, history[0], c='blue')
    ax.set_xlim(0.0, 1.0)
    A = max(np.max(np.abs(s)) for s in history)
    ax.set_ylim(-1.2*A, 1.2*A)
    ax.set_xlabel('x')
    ax.set_ylabel('u(x,t)')
    ax.set_title(title)
    ax.grid(True)
    
    def update(frame):
        line.set_ydata(history[frame])
        return line,
    
    anim = animation.FuncAnimation(fig, update, frames=len(history), interval=100)
    html = anim.to_jshtml()
    return HTML(html)


In [ ]:
r_values = [0.5, 1.0, 1.05]
Nx = 151
T = 0.6
half_width = 0.08

for r in r_values:
    x, snaps = simulate_wave(r, L=1.0, c=1.0, Nx=Nx, T=T, half_width=half_width, max_frames=120)
    display(make_animation(x, snaps, title=f"Решение одномерного волнового уравнения, CFL={r:.2f}"))

При CFL > 1 схема неустойчива.

## Продвинутый уровень

Реализуем граничные условия Неймана:

$$\frac{\partial u}{\partial t}(t, 0) = 0, \quad \frac{\partial u}{\partial t}(t, L) = 0.$$

Дискретизируется в виде:

$$\frac{u_2 - u_1}{\Delta t} = 0, \quad \frac{u_N - u_{N-1}}{\Delta t} = 0,$$

$$u_1 = u_2, \quad u_N = u_{N-1}.$$

In [ ]:
def apply_bc_values(u, bc):
    left, right = bc
    if left == 'dirichlet':
        u[0] = 0.0
    elif left == 'neumann':
        u[0] = u[1]
    if right == 'dirichlet':
        u[-1] = 0.0
    elif right == 'neumann':
        u[-1] = u[-2]
    return u

def energy(u_now, u_prev, dx, c, dt):
    v = (u_now - u_prev) / dt
    kin = np.sum(v**2) * dx
    ux = (u_now[1:] - u_now[:-1]) / dx
    pot = np.sum((c**2) * ux**2) * dx
    return 0.5 * (kin + pot)

def simulate_wave(r, L=1.0, c=1.0, Nx=151, T=0.6, half_width=0.08, u0_func=None, v0_func=None, bc=('dirichlet', 'neumann'), max_frames=120):
    dx = L / (Nx - 1)
    dt = r * dx / c
    Nt = int(np.ceil(T / dt))
    x = np.linspace(0, L, Nx)
    u0 = u0_func(x)
    u_prev = u0.copy()
    u_curr = u_prev.copy()
    r2 = (c * dt / dx) ** 2

    times = [0.0]
    energies = [energy(u_prev, u_curr, dx, c, dt)]

    if u0_func is None:
        u0 = np.zeros_like(x)
    else:
        u0 = u0_func(x)
    if v0_func is None:
        v0 = np.zeros_like(x)
    else:
        v0 = v0_func(x)
    
    u_prev = apply_bc_values(u_prev, bc)
    for i in range(1, Nx-1):
        u_curr[i] = u_prev[i] + dt * v0[i] + 0.5 * r2 * (u_prev[i+1] - 2 * u_prev[i] + u_prev[i-1])
    u_curr = apply_bc_values(u_curr, bc)
    
    save_every = max(1, int(np.ceil(Nt / max_frames)))
    
    history = [u_prev.copy()]

    for n in range(1, Nt):
        u_curr = apply_bc_values(u_curr, bc)
        u_next = np.zeros_like(u_curr)
        for i in range(1, Nx-1):
            u_next[i] = 2*u_curr[i] - u_prev[i] + r2 * (u_curr[i+1] - 2*u_curr[i] + u_curr[i-1])
        u_next = apply_bc_values(u_next, bc)
        t = (n+1) * dt
        if n % save_every == 0:
            history.append(u_next.copy())
            times.append(t)
            energies.append(energy(u_next, u_curr, dx, c, dt))
        u_prev, u_curr = u_curr, u_next
    return x, history, times, energies



In [ ]:
L = 1.0
c = 1.0
Nx = 201
r = 0.9
T = 1.2

def gauss(x, x0=0.2, sigma=0.04):
    return np.exp(-0.5*((x-x0)/sigma)**2)

x, history, times, energies = simulate_wave(r, L=L, c=c, Nx=Nx, T=T, u0_func=lambda x: gauss(x,0.2,0.04), v0_func=lambda x: np.zeros_like(x), bc=('neumann','neumann'), max_frames=240)
display(make_animation(x, history, title="Граничные условия Неймана"))

In [ ]:
plt.figure(figsize=(7,3))
plt.plot(times, energies, marker='o', markersize=3, linewidth=1, c='blue')
plt.xlabel('t')
plt.ylabel('E(t)')
plt.title('Энергия системы')
plt.grid(True)
plt.show()

print('Энергия системы сохраняется с относительной точностью {}%.'.format(np.max((energies-energies[0])/energies[0]*100)))

Смоделируем стоячую волну, задав в качестве начального условия $\sin \left(\frac{n \pi x}{L}\right)$.

In [ ]:
L = 1.0
c = 1.0
Nx = 201
r = 0.9
T = 1.2

x, history, times, energies = simulate_wave(r, L=L, c=c, Nx=Nx, T=T, u0_func=lambda x: np.sin(5*np.pi*x/L), v0_func=lambda x: np.zeros_like(x), bc=('dirichlet','dirichlet'), max_frames=240)
display(make_animation(x, history, title="Стоячая волна sin(5πx/L)"))